# Women's E-Commerce Clothing Reviews: Data Cleaning and EDA

**Dataset:** Women's E-Commerce Clothing Reviews (~23.5k customer reviews with rating, recommendation flag, age and product category).
**Used for:** review / editorial summarization.
**Goal:** clean the data (missing values, inconsistent text, duplicates, typos, outliers), explore it and save `data/processed/womens_reviews_cleaned.csv`.

> Run on Kaggle with the dataset added as input, or locally with `Womens Clothing E-Commerce Reviews.csv` in `data/raw/`. The first code cell picks the right paths automatically.

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

RAW_NAME = "Womens Clothing E-Commerce Reviews.csv"

# Works both locally (run from the notebooks/ folder) and on Kaggle
kaggle_hits = sorted(Path("/kaggle/input").glob(f"**/{RAW_NAME}")) if Path("/kaggle/input").exists() else []
if kaggle_hits:
    RAW_PATH = kaggle_hits[0]
    OUT_DIR  = Path("/kaggle/working/data/processed")
    FIG_DIR  = Path("/kaggle/working/images/womens_reviews")
else:
    RAW_PATH = Path("../data/raw") / RAW_NAME     # download from Kaggle and put the CSV here
    OUT_DIR  = Path("../data/processed")
    FIG_DIR  = Path("../images/womens_reviews")
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Reading:", RAW_PATH)

## Load the data

In [ ]:
df = pd.read_csv(RAW_PATH)
rows_raw = len(df)
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

## Column names
Convert column names to `snake_case` (e.g. `Review Text` -> `review_text`) and drop `Unnamed: 0`, which is just the old row index.

In [ ]:
df.columns = [re.sub(r"[^0-9a-z]+", "_", c.strip().lower()).strip("_") for c in df.columns]
df = df.drop(columns=[c for c in df.columns if c.startswith("unnamed")])
print(df.columns.tolist())

## Missing values

In [ ]:
missing = df.isna().sum()
print(pd.DataFrame({"missing": missing, "percent": (missing / len(df) * 100).round(2)}))

- **`review_text` missing:** these rows are dropped. The dataset is used for review summarization, so a row without review text is useless.
- **`title` missing:** many reviews have no title, but the review itself is still useful, so the gap is filled with `no title`.
- **`division_name`, `department_name`, `class_name` missing:** only a handful of rows; filled with `Unknown` instead of being dropped.

In [ ]:
before = len(df)
df = df.dropna(subset=["review_text"])
print("Rows dropped (no review text):", before - len(df))

df["title"] = df["title"].fillna("no title")
for c in ["division_name", "department_name", "class_name"]:
    df[c] = df[c].fillna("Unknown")

print("Missing values left:", df.isna().sum().sum())

## Format fixes
- Strip leading/trailing spaces from all text columns.
- Normalise Windows line breaks (`\r\n`) in the review text to `\n`.
- Fix the typo `Initmates` -> `Intimates` in `division_name` (it is in the original data).

In [ ]:
text_cols = ["title", "review_text", "division_name", "department_name", "class_name"]
for c in text_cols:
    df[c] = df[c].astype(str).str.strip()

df["review_text"] = df["review_text"].str.replace("\r\n", "\n", regex=False)
df["division_name"] = df["division_name"].replace("Initmates", "Intimates")

for c in ["division_name", "department_name"]:
    print(df[c].value_counts(), "\n")

## Duplicates
Check for fully repeated rows and for the same review text appearing more than once (the same review posted twice). Keep the first copy.

In [ ]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Repeated review texts:", df["review_text"].duplicated().sum())

before = len(df)
df = df.drop_duplicates(subset=["review_text"], keep="first")
print("Rows dropped (duplicates):", before - len(df))

## Invalid values
Ratings must be 1-5, `recommended_ind` must be 0/1, ages must be adults (18+), feedback counts can't be negative.

In [ ]:
valid = (
    df["rating"].between(1, 5)
    & df["recommended_ind"].isin([0, 1])
    & (df["age"] >= 18)
    & (df["positive_feedback_count"] >= 0)
)
print("Invalid rows:", (~valid).sum())
df = df[valid]

## Outliers
Ages above the IQR limit are **flagged** in `is_age_outlier`, not removed. They may be real customers, and the review text is still useful.
`review_text_length` (characters) is added for EDA; the site limits reviews to about 500 characters, so there are no extreme lengths.

In [ ]:
q1, q3 = df["age"].quantile([0.25, 0.75])
age_limit = q3 + 1.5 * (q3 - q1)
df["is_age_outlier"] = df["age"] > age_limit
print("Age IQR upper limit:", age_limit)
print("Flagged ages:", df["is_age_outlier"].sum())

df["review_text_length"] = df["review_text"].str.len()
print(df["review_text_length"].describe())

## EDA

In [ ]:
def save(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / name, dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(x="rating", data=df, color="#4C72B0")
plt.title("Distribution of Customer Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of reviews")
save("rating_distribution.png")

**What the graph shows:** ratings are strongly skewed towards positive: about 55% of reviews are 5 stars and only about 10% are 1-2 stars. A summarizer trained on all reviews will mostly see positive text, so negative reviews may need to be sampled separately.

In [ ]:
plt.figure(figsize=(6, 4))
counts = df["recommended_ind"].map({1: "Recommended", 0: "Not recommended"}).value_counts()
sns.barplot(x=counts.index, y=counts.values, color="#4C72B0")
plt.title("Would the Customer Recommend the Product?")
plt.xlabel("")
plt.ylabel("Number of reviews")
save("recommendation_distribution.png")
print((df["recommended_ind"].mean() * 100).round(1), "% recommend")

**What the graph shows:** about 82% of reviewers would recommend the product. The two classes are imbalanced (roughly 4.5 : 1), which matters if `recommended_ind` is used as a sentiment label.

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["age"], bins=40, color="#4C72B0")
plt.axvline(age_limit, color="red", linestyle="--", label=f"IQR limit ({age_limit:.0f})")
plt.title("Age Distribution of Reviewers")
plt.xlabel("Age")
plt.ylabel("Number of reviews")
plt.legend()
save("age_distribution.png")
print(df["age"].describe())

**What the graph shows:** most reviewers are 30-55 (median 41). The distribution has a long right tail; ages above the red IQR limit (79) are flagged in `is_age_outlier`, not removed.

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["review_text_length"], bins=40, color="#4C72B0")
plt.title("Review Text Length")
plt.xlabel("Characters")
plt.ylabel("Number of reviews")
save("review_text_length.png")

**What the graph shows:** review lengths are spread fairly evenly between about 50 and 480 characters, with a large spike just under 500, because the site caps reviews at about 500 characters. Around a fifth of reviews hit the cap, so long reviews may be cut off. All reviews are short enough to feed into a summarization model in one piece.

In [ ]:
dept = df.groupby("department_name")["rating"].agg(["mean", "count"]).sort_values("mean", ascending=False)
print(dept)
dept = dept.drop(index="Unknown", errors="ignore")   # only a handful of reviews, would distort the chart
plt.figure(figsize=(8, 4))
ax = sns.barplot(x=dept.index, y=dept["mean"], color="#4C72B0")
ax.bar_label(ax.containers[0], fmt="%.2f")
plt.ylim(0, 5)
plt.title("Average Rating by Department")
plt.xlabel("Department")
plt.ylabel("Average rating")
save("average_rating_by_department.png")

**What the graph shows:** average ratings are very similar across departments (about 4.1-4.3). Bottoms, Intimate and Jackets score highest; Trend scores lowest (about 3.8) but has only ~120 reviews. Department alone says little about how positive a review is.

In [ ]:
num_cols = ["age", "rating", "recommended_ind", "positive_feedback_count", "review_text_length"]
plt.figure(figsize=(7, 5))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Between Numeric Review Features")
save("numeric_correlation.png")

**What the graph shows:** `rating` and `recommended_ind` are strongly correlated (0.79), so they carry largely the same information. Age and review length are almost uncorrelated with rating; longer reviews get slightly more positive feedback (0.19). The text itself, not the metadata, carries the signal.

## Final check and save

In [ ]:
df = df.reset_index(drop=True)
assert df.isna().sum().sum() == 0
assert df["review_text"].is_unique
print(f"Rows: {rows_raw:,} raw -> {len(df):,} clean")
print("Columns:", df.columns.tolist())

df.to_csv(OUT_DIR / "womens_reviews_cleaned.csv", index=False)
print("Saved:", OUT_DIR / "womens_reviews_cleaned.csv")